# 10 — Audit the frozen V3 ensemble

This stage loads the already-frozen five-fold models and thresholds, then
evaluates the original seven test subjects once. Because those subjects were
inspected during V2 development, OOF performance remains the primary estimate;
this result is an audit comparison rather than a pristine final test.

In [1]:
from pathlib import Path
import os
import sys


def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists() and (candidate / "apps").exists():
            return candidate
    raise RuntimeError("Run this notebook inside the pure-pulsesure repository")


REPO_ROOT = find_repository_root(Path.cwd().resolve())
RAW_DIR = Path(os.environ.get("PURE_BUT_PPG_DIR", REPO_ROOT / "data" / "raw" / "but-ppg-2.0.0")).expanduser().resolve()
V3_DATA_DIR = Path(os.environ.get("PURE_V3_DATA_DIR", REPO_ROOT / "data" / "processed" / "but-ppg-v3")).expanduser().resolve()
V3_MODEL_DIR = Path(os.environ.get("PURE_V3_MODEL_DIR", REPO_ROOT / "models" / "production" / "v3")).expanduser().resolve()

print("Raw:", RAW_DIR)
print("V3 data:", V3_DATA_DIR)
print("V3 models:", V3_MODEL_DIR)

Raw: /mnt/home/gracewahyuni06/pure-pulsesure/data/raw/but-ppg-2.0.0
V3 data: /mnt/home/gracewahyuni06/pure-pulsesure/data/processed/but-ppg-v3
V3 models: /mnt/home/gracewahyuni06/pure-pulsesure/models/production/v3


In [2]:
import subprocess

subprocess.run(
    [
        sys.executable,
        str(REPO_ROOT / "scripts" / "evaluate_v3_ensemble.py"),
        "--input-dir", str(V3_DATA_DIR),
        "--model-dir", str(V3_MODEL_DIR),
    ],
    check=True,
)

{
  "test_binary": {
    "threshold": 0.23500000000000001,
    "accuracy": 0.7375,
    "balanced_accuracy": 0.7461827284105131,
    "precision_good": 0.5038759689922481,
    "recall_good_sensitivity": 0.7647058823529411,
    "specificity_poor": 0.7276595744680852,
    "f1_good": 0.6074766355140186,
    "false_acceptance_rate": 0.2723404255319149,
    "false_rejection_rate": 0.23529411764705882,
    "roc_auc": 0.8247058823529412,
    "average_precision": 0.6069637318737133,
    "confusion_matrix": [
      [
        342,
        128
      ],
      [
        40,
        130
      ]
    ]
  },
  "test_gate": {
    "lower_threshold": 0.07319095477386936,
    "upper_threshold": 0.41067839195979905,
    "false_acceptance_rate": 0.17659574468085107,
    "false_rejection_rate": 0.029411764705882353,
    "good_acceptance_rate": 0.6823529411764706,
    "poor_rejection_rate": 0.3191489361702128,
    "reliable_coverage": 0.3109375,
    "caution_rate": 0.446875,
    "unreliable_rate": 0.2421875
  },

CompletedProcess(args=['/mnt/home/gracewahyuni06/pure-pulsesure/.venv/bin/python', '/mnt/home/gracewahyuni06/pure-pulsesure/scripts/evaluate_v3_ensemble.py', '--input-dir', '/mnt/home/gracewahyuni06/pure-pulsesure/data/processed/but-ppg-v3', '--model-dir', '/mnt/home/gracewahyuni06/pure-pulsesure/models/production/v3'], returncode=0)

In [3]:
import json
import pandas as pd

report = json.loads((V3_MODEL_DIR / "test_metrics.json").read_text())
display(pd.Series(report["test_binary"]).to_frame("audit test"))
display(pd.Series(report["test_gate"]).to_frame("audit gate"))
display(pd.DataFrame(report["branch_test_metrics"]).T[["accuracy", "balanced_accuracy", "f1_good", "roc_auc", "false_acceptance_rate"]])
print("Reliable precision:", report["reliable_precision"])

,audit test
threshold,0.235
accuracy,0.7375
balanced_accuracy,0.746183
precision_good,0.503876
recall_good_sensitivity,0.764706
specificity_poor,0.72766
f1_good,0.607477
false_acceptance_rate,0.27234
false_rejection_rate,0.235294
roc_auc,0.824706


,audit gate
lower_threshold,0.073191
upper_threshold,0.410678
false_acceptance_rate,0.176596
false_rejection_rate,0.029412
good_acceptance_rate,0.682353
poor_rejection_rate,0.319149
reliable_coverage,0.310937
caution_rate,0.446875
unreliable_rate,0.242188


,accuracy,balanced_accuracy,f1_good,roc_auc,false_acceptance_rate
waveform,0.732812,0.733605,0.593824,0.80622,0.268085
stft,0.75,0.730288,0.593909,0.791151,0.22766
sqi,0.770312,0.764768,0.635236,0.826471,0.223404


Reliable precision: 0.5829145728643216
